# Capítulo 12 — Aprendizado Profundo Não Supervisionado: Autoencoders, VAEs, GANs e Autossupervisão

Códigos do capítulo. Execute as células em ordem; cada bloco é independente dos demais.


## 12.3.4 Implementação em PyTorch

In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler

torch.manual_seed(0)

# 1) Dados: dígitos 8x8 (64 atributos) escalados em [0,1]
X, y = load_digits(return_X_y=True)
X = MinMaxScaler().fit_transform(X).astype("float32")
Xt = torch.from_numpy(X)

# 2) Modelo: 64 -> 32 -> 2 (código) -> 32 -> 64
class Autoencoder(nn.Module):
    def __init__(self, dim_latente=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(),
                                 nn.Linear(32, dim_latente))
        self.dec = nn.Sequential(nn.Linear(dim_latente, 32), nn.ReLU(),
                                 nn.Linear(32, 64), nn.Sigmoid())
    def forward(self, x):
        z = self.enc(x)
        return self.dec(z), z

modelo = Autoencoder(dim_latente=2)
otimizador = torch.optim.Adam(modelo.parameters(), lr=1e-2)
perda_fn = nn.MSELoss()

# 3) Treino
for epoca in range(150):
    rec, z = modelo(Xt)
    perda = perda_fn(rec, Xt)
    otimizador.zero_grad()
    perda.backward()
    otimizador.step()
    if epoca % 50 == 0:
        print(f"época {epoca:3d}  perda = {perda.item():.4f}")

# 4) Usar o espaço latente (2D) para visualizar os dígitos
with torch.no_grad():
    Z = modelo.enc(Xt).numpy()
print("Formato do espaço latente:", Z.shape)

## 12.4.3 Autoencoder de remoção de ruído (*denoising*)

In [ ]:
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler

torch.manual_seed(0)
X = MinMaxScaler().fit_transform(load_digits().data).astype("float32")
limpo = torch.from_numpy(X)

dae = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(),
                    nn.Linear(16, 32), nn.ReLU(), nn.Linear(32, 64), nn.Sigmoid())
opt = torch.optim.Adam(dae.parameters(), lr=5e-3)

for epoca in range(200):
    ruidoso = (limpo + 0.3 * torch.randn_like(limpo)).clamp(0, 1)   # corrompe
    perda = nn.functional.mse_loss(dae(ruidoso), limpo)             # alvo = versão limpa
    opt.zero_grad(); perda.backward(); opt.step()

with torch.no_grad():
    teste = (limpo + 0.3 * torch.randn_like(limpo)).clamp(0, 1)
    print("Erro da imagem ruidosa    :", nn.functional.mse_loss(teste, limpo).item())
    print("Erro da imagem recuperada :", nn.functional.mse_loss(dae(teste), limpo).item())

## 12.4.5 Aplicações

In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score

torch.manual_seed(0)
X, y = load_digits(return_X_y=True)
normais, anomalias = X[y != 9], X[y == 9]          # o dígito 9 é a "novidade"

# treino / validação / teste: só com exemplos normais (as anomalias ficam para o teste)
tr, resto = train_test_split(normais, test_size=0.4, random_state=0)
va, te_normal = train_test_split(resto, test_size=0.5, random_state=0)

esc = MinMaxScaler().fit(tr)                        # escalonador ajustado SÓ no treino
t = lambda a: torch.tensor(esc.transform(a), dtype=torch.float32)
tr_t, va_t, ten_t, tea_t = t(tr), t(va), t(te_normal), t(anomalias)

ae = nn.Sequential(nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(),
                   nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 64), nn.Sigmoid())
opt = torch.optim.Adam(ae.parameters(), lr=5e-3)
for _ in range(300):
    perda = nn.functional.mse_loss(ae(tr_t), tr_t)
    opt.zero_grad(); perda.backward(); opt.step()

def erro(x):
    with torch.no_grad():
        return ((ae(x) - x) ** 2).mean(1).numpy()

limiar = np.percentile(erro(va_t), 95)              # limiar escolhido SÓ na validação

e_teste = np.r_[erro(ten_t), erro(tea_t)]
y_teste = np.r_[np.zeros(len(ten_t)), np.ones(len(tea_t))]
alerta = e_teste > limiar
print("AUC-ROC             :", round(roc_auc_score(y_teste, e_teste), 3))
print("Precisão no limiar  :", round(precision_score(y_teste, alerta), 3))
print("Revocação no limiar :", round(recall_score(y_teste, alerta), 3))

## 12.5.4 VAE em PyTorch

In [ ]:
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler

torch.manual_seed(0)
X = MinMaxScaler().fit_transform(load_digits().data).astype("float32")
Xt = torch.from_numpy(X)

class VAE(nn.Module):
    def __init__(self, dim_z=2):
        super().__init__()
        self.enc = nn.Sequential(nn.Linear(64, 32), nn.ReLU())
        self.mu = nn.Linear(32, dim_z)
        self.logvar = nn.Linear(32, dim_z)          # log da variância (mais estável)
        self.dec = nn.Sequential(nn.Linear(dim_z, 32), nn.ReLU(),
                                 nn.Linear(32, 64), nn.Sigmoid())
    def forward(self, x):
        h = self.enc(x)
        mu, logvar = self.mu(h), self.logvar(h)
        eps = torch.randn_like(mu)
        z = mu + torch.exp(0.5 * logvar) * eps      # reparametrização
        return self.dec(z), mu, logvar

def perda_vae(rec, x, mu, logvar):
    r = nn.functional.binary_cross_entropy(rec, x, reduction="sum")
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return (r + kl) / x.shape[0]

vae = VAE()
opt = torch.optim.Adam(vae.parameters(), lr=5e-3)
for epoca in range(200):
    rec, mu, logvar = vae(Xt)
    perda = perda_vae(rec, Xt, mu, logvar)
    opt.zero_grad(); perda.backward(); opt.step()
print("Perda final por exemplo:", round(perda.item(), 2))

# GERAR dígitos novos: sorteie z ~ N(0,1) e decodifique
with torch.no_grad():
    novos = vae.dec(torch.randn(16, 2))
print("Imagens geradas:", novos.shape)

## 12.6.4 Um esboço em PyTorch

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
# Dados "reais": uma distribuição 1D bimodal (duas corcovas)
def reais(n):
    metade = n // 2
    return torch.cat([torch.randn(metade, 1) * 0.5 - 2, torch.randn(n - metade, 1) * 0.5 + 2])

G = nn.Sequential(nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 1))
D = nn.Sequential(nn.Linear(1, 16), nn.ReLU(), nn.Linear(16, 1), nn.Sigmoid())
optG = torch.optim.Adam(G.parameters(), lr=2e-3)
optD = torch.optim.Adam(D.parameters(), lr=2e-3)
bce = nn.BCELoss()

for passo in range(1500):
    # 1) treina o discriminador: reais -> 1, falsos -> 0
    real = reais(128)
    falso = G(torch.randn(128, 4)).detach()
    perdaD = bce(D(real), torch.ones(128, 1)) + bce(D(falso), torch.zeros(128, 1))
    optD.zero_grad(); perdaD.backward(); optD.step()
    # 2) treina o gerador: quer que o discriminador diga "1" para os falsos
    perdaG = bce(D(G(torch.randn(128, 4))), torch.ones(128, 1))
    optG.zero_grad(); perdaG.backward(); optG.step()

with torch.no_grad():
    amostra = G(torch.randn(2000, 4)).flatten()
print("Média do gerado:", amostra.mean().item(), "| desvio:", amostra.std().item())
frac = (amostra < 0).float().mean().item()
print("Fração negativa (próxima de 0,5 se houver duas corcovas):", frac)

## 12.9.4 Um exemplo mínimo

In [ ]:
import torch
import torch.nn.functional as F

def nt_xent(z1, z2, tau=0.5):
    """Perda contrastiva para um lote: z1[i] e z2[i] são duas visões do mesmo exemplo."""
    z = F.normalize(torch.cat([z1, z2]), dim=1)          # normaliza (cosseno)
    sim = z @ z.T / tau                                   # similaridades
    n = z1.shape[0]
    sim.fill_diagonal_(float("-inf"))                     # ignora a similaridade consigo mesmo
    alvo = torch.cat([torch.arange(n, 2 * n), torch.arange(0, n)])   # o positivo de cada um
    return F.cross_entropy(sim, alvo)

torch.manual_seed(0)
base = torch.randn(8, 16)
visao1 = base + 0.05 * torch.randn(8, 16)                 # duas "visões" quase iguais
visao2 = base + 0.05 * torch.randn(8, 16)
aleatorio = torch.randn(8, 16)

print("Perda com positivos parecidos:", nt_xent(visao1, visao2).item())
print("Perda com pares aleatórios   :", nt_xent(visao1, aleatorio).item())

## 12.10 Usando representações aprendidas

In [ ]:
import numpy as np
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

torch.manual_seed(0)
X, y = load_digits(return_X_y=True)
X = MinMaxScaler().fit_transform(X).astype("float32")
Xt = torch.from_numpy(X)

enc = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 10))
dec = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 64), nn.Sigmoid())
opt = torch.optim.Adam(list(enc.parameters()) + list(dec.parameters()), lr=5e-3)
for _ in range(300):
    perda = nn.functional.mse_loss(dec(enc(Xt)), Xt)
    opt.zero_grad(); perda.backward(); opt.step()

with torch.no_grad():
    Z = enc(Xt).numpy()

ari_pixels = adjusted_rand_score(y, KMeans(10, n_init=10, random_state=0).fit_predict(X))
ari_latente = adjusted_rand_score(y, KMeans(10, n_init=10, random_state=0).fit_predict(Z))
print("ARI K-means nos pixels  :", round(ari_pixels, 3))
print("ARI K-means no latente  :", round(ari_latente, 3))

## Resumo do capítulo